# P1: Cloud ML Data Pipeline

**Module 3 — Week 10: MLOps & Production Systems**

## Problem Statement

Build a local simulation of a cloud-based ML data pipeline that mirrors production S3 workflows.
We will ingest raw customer churn data, validate it, preprocess it, engineer features,
and store results across organized S3 prefixes — all using a local simulator that mimics
the AWS S3 API.

## Metrics

- **Data quality**: null rates, duplicate counts, validation pass/fail
- **Pipeline reliability**: end-to-end success rate, stage timing
- **Storage efficiency**: CSV vs Parquet size comparison
- **Throughput**: rows processed per second across pipeline stages

---
## 1. Setup and Configuration

In [ ]:
import sys
import os
import json
import time
from datetime import datetime
from io import BytesIO, StringIO

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path so we can import src modules
sys.path.insert(0, '../../..')

from src.generate_data import generate_churn_data
from src.s3_pipeline import LocalS3Simulator, validate_data, preprocess, run_pipeline

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

In [ ]:
# Initialize the local S3 simulator
s3 = LocalS3Simulator()

# Create the bucket for our churn pipeline
s3.create_bucket(Bucket='churn-pipeline')

print('S3 simulator initialized.')
print(f'Bucket created: churn-pipeline')
print(f'Timestamp: {datetime.now().isoformat()}')

---
## 2. Generate and Explore Data

In [ ]:
# Generate the 8000-row churn dataset
df = generate_churn_data()

In [ ]:
# TODO: Print shape, dtypes, and first few rows
# print(df.shape)
# print(df.dtypes)
# df.head()

In [ ]:
# TODO: Check null rates per column
# Hint: df.isnull().mean().sort_values(ascending=False)


In [ ]:
# TODO: Check duplicate count
# Hint: df.duplicated().sum()


In [ ]:
# TODO: Plot churn distribution
# Hint: Use value_counts() and plot a bar chart
# - Show counts and percentages
# - Add a title: "Churn Distribution in Raw Data"


---
## 3. Upload Raw Data to S3

In [ ]:
# TODO: Convert DataFrame to CSV bytes and upload to S3
# Key: raw/churn/2024-01.csv
#
# Steps:
# 1. Convert df to CSV string, then encode to bytes
# 2. Use s3.put_object(Bucket='churn-pipeline', Key='raw/churn/2024-01.csv', Body=csv_bytes)


In [ ]:
# TODO: Upload metadata JSON to raw/churn/metadata.json
# The metadata should include:
# - schema: dict of column name -> dtype (as string)
# - timestamp: current ISO timestamp
# - row_count: number of rows
#
# Steps:
# 1. Build the metadata dict
# 2. json.dumps() and encode to bytes
# 3. s3.put_object(...)


In [ ]:
# TODO: Verify upload with list_objects_v2
# Hint: s3.list_objects_v2(Bucket='churn-pipeline', Prefix='raw/')
# Print each object key and size


---
## 4. Data Validation

In [ ]:
# TODO: Download raw data from S3 and run validate_data()
#
# Steps:
# 1. response = s3.get_object(Bucket='churn-pipeline', Key='raw/churn/2024-01.csv')
# 2. Parse CSV bytes back into a DataFrame
# 3. validation_report = validate_data(df_downloaded)


In [ ]:
# TODO: Print the validation report
# Show each check and its pass/fail status


In [ ]:
# TODO: Handle both pass and fail scenarios
# - If validation passed: print success and proceed
# - If validation failed: print which checks failed and what action to take
#
# Save the validation report to logs/ prefix in S3
# Key: logs/validation/2024-01-report.json


---
## 5. Preprocessing (Lambda Simulation)

In production, an S3 event (e.g., `s3:ObjectCreated:*`) would trigger a Lambda function
to automatically preprocess newly uploaded data. Here we simulate that by manually
calling the `preprocess()` function after detecting validated data in our bucket.

In [ ]:
# TODO: Download the validated data from S3
# (Use the raw data we uploaded, since it passed validation)


In [ ]:
# TODO: Run preprocess() on the data
# preprocess() removes duplicates, imputes nulls, and clips ranges
#
# df_processed = preprocess(df_raw)


In [ ]:
# TODO: Compare before/after stats
# - Row count before vs after (duplicates removed)
# - Null counts before vs after
# - Descriptive stats comparison for numeric columns


In [ ]:
# TODO: Save processed data to S3 as Parquet
# Key: processed/churn/2024-01.parquet
#
# Steps:
# 1. Write DataFrame to a BytesIO buffer as Parquet
#    buf = BytesIO()
#    df_processed.to_parquet(buf, index=False)
# 2. Upload buf.getvalue() to S3


---
## 6. Feature Engineering

In [ ]:
# TODO: Create derived features from the processed data
#
# 1. tenure_bucket: categorize tenure_months into Short (<12), Medium (12-36), Long (>36)
#    Hint: pd.cut() or np.select()
#
# 2. charge_per_month: total_charges / tenure_months
#    Handle division by zero (tenure_months == 0)
#
# 3. is_high_value: True if monthly_charges > 75th percentile
#    Hint: df['monthly_charges'].quantile(0.75)
#
# 4. risk_score: composite score from churn risk factors
#    Example formula: combine normalized tenure (inverse), contract type,
#    monthly charges, and number of support tickets
#    Scale to 0-100 range


In [ ]:
# TODO: Inspect the new features
# - Print value counts for tenure_bucket
# - Print summary stats for charge_per_month, risk_score
# - Print count/percentage of is_high_value


In [ ]:
# TODO: Save feature-engineered data to S3
# Key: features/churn/2024-01.parquet


---
## 7. Pipeline Orchestration

In [ ]:
# TODO: Use run_pipeline() to chain all stages end-to-end
# run_pipeline() orchestrates: generate -> validate -> preprocess -> store
#
# result = run_pipeline(s3, bucket='churn-pipeline')


In [ ]:
# TODO: Print the pipeline result with timing
# - Overall status (success/failure)
# - Time taken per stage
# - Total pipeline duration


In [ ]:
# TODO: Run the pipeline again with fresh data to show consistency
# - Generate new data and run pipeline
# - Compare results across runs


---
## 8. Pipeline Monitoring

In [ ]:
# TODO: Track multiple pipeline runs
# Run the pipeline 5 times and collect timing metrics for each run
#
# run_metrics = []
# for i in range(5):
#     result = run_pipeline(s3, bucket='churn-pipeline')
#     run_metrics.append(result)


In [ ]:
# TODO: Create a summary DataFrame with timing per stage
# Columns: run_id, stage_name, duration_seconds, status
# Print the summary table


In [ ]:
# TODO: Plot stage durations across runs
# - Bar chart or box plot showing time distribution per stage
# - Add title: "Pipeline Stage Durations Across Runs"


In [ ]:
# TODO: Identify bottlenecks
# - Which stage takes the longest on average?
# - What is the variance in each stage?
# - Print a ranked list of stages by mean duration


---
## 9. What I Built / What I Learned

In [ ]:
# Summary of bucket contents
# TODO: List all objects in the churn-pipeline bucket with their sizes
#
# response = s3.list_objects_v2(Bucket='churn-pipeline')
# for obj in response.get('Contents', []):
#     print(f"  {obj['Key']:50s}  {obj['Size']:>10,} bytes")


In [ ]:
# TODO: Compare CSV input vs Parquet output sizes
# - Get size of raw/churn/2024-01.csv
# - Get size of processed/churn/2024-01.parquet
# - Calculate compression ratio and space savings percentage


### Key Learnings

**TODO: Fill in your answers below after completing the project.**

1. **Most important S3 pattern learned:**  
   _Your answer here_

2. **Data quality issues found:**  
   _Your answer here_

3. **Pipeline bottleneck:**  
   _Your answer here_

4. **What I'd add for production:**  
   _Your answer here_

5. **CSV vs Parquet savings:**  
   _Your answer here_